# 02 — Preprocessing and Stratified Split

This notebook applies the paper's cleaning steps and creates a stratified 90:10 train/validation split of the official ArSarcasm-v2 training file. The official test file is **only preprocessed and saved**; its labels are never used for splitting, tuning, or model selection.

The preprocessing function is `src/bonus/preprocessing.py::preprocess_text`. The split and label mapping live in `src/bonus/data.py`. The notebook only calls those modules and displays their output.

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Walk upwards until the repository root (the folder that contains data/raw) is found.
ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "raw").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from bonus import data, eda, preprocessing, utils

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
print("Repository root:", ROOT)

Repository root: /home/nour/FCAI-H/fourth year last_dance/Sem1/Neural Networks & deep learning/project


## 1. Preprocessing sanity checks

The table below is an assertion-backed set of before/after examples. It covers URLs (including hashtags glued to URLs), mentions, hashtags, repeated characters, digits, English letters, brackets, emoticons, extra spaces, HTML, email addresses, tashkeel, tatweel, Fitzpatrick skin-tone modifiers, Arabic presentation forms, and slash replacement. It also asserts that the fixed emoticon and HTML patterns preserve text such as `[الزخرف:18]`, `Naxos`, `Dell XPS 13`, `SA8*`, and `m8S`. The cell raises an error if any expected output does not match.

In [2]:
checks = preprocessing.sanity_check_table()
display(checks)
print(f"{len(checks)} preprocessing checks passed.")

,case,input,expected,actual,passed
0,URL,شاهد الفيديو https://t.co/abc123 الآن,شاهد الفيديو [رابط] الآن,شاهد الفيديو [رابط] الآن,True
1,Mention,مرحبا @user_1 كيف حالك,مرحبا [مستخدم] كيف حالك,مرحبا [مستخدم] كيف حالك,True
2,Hashtag,هذا #فلول مهم,هذا #فلول مهم,هذا #فلول مهم,True
3,Hashtag with Latin/digits,#COVID19 منتشر,# COVID 19 منتشر,# COVID 19 منتشر,True
4,Repeated characters,هههههههه والله,هه والله,هه والله,True
5,Digits,عام2021 ثم 2022عام,عام 2021 ثم 2022 عام,عام 2021 ثم 2022 عام,True
6,English letters,هذاiPhoneجديد,هذا iPhone جديد,هذا iPhone جديد,True
7,Brackets,نص[مهم]جدا,نص [ مهم ] جدا,نص [ مهم ] جدا,True
8,Emoticons,أحبك :) 😍,أحبك,أحبك,True
9,Extra spaces,هذا نص,هذا نص,هذا نص,True


29 preprocessing checks passed.


## 2. Load the config and build the split

The seed comes from `configs/bonus/split.yaml`. The split is stratified on the sentiment label. The test file is loaded only for preprocessing and saving.

In [3]:
config = data.load_config(ROOT / "configs/bonus/split.yaml")
print("Config:")
for key, value in config.items():
    print(f"  {key}: {value}")

utils.set_seed(config["seed"])

splits = data.build_splits(
    raw_train_path=ROOT / config["train_file"],
    raw_test_path=ROOT / config["test_file"],
    seed=config["seed"],
    validation_fraction=config["validation_fraction"],
)

for name in ("train", "validation", "test"):
    print(f"{name}: shape={splits[name].shape}, columns={list(splits[name].columns)}")

Config:
  seed: 42
  validation_fraction: 0.1
  train_file: data/raw/arsarcasm_v2/training_data.csv
  test_file: data/raw/arsarcasm_v2/testing_data.csv
  output_dir: data/splits/arsarcasm_v2


train: shape=(11293, 2), columns=['text', 'label']
validation: shape=(1255, 2), columns=['text', 'label']
test: shape=(3000, 2), columns=['text', 'label']


### Fixed label mapping

In [4]:
display(data.label_mapping_table())

,label,label_id
0,NEG,0
1,NEU,1
2,POS,2


### Split sizes and class counts compared with the paper

The paper reports 11,293 training rows and 1,255 validation rows. Its Table 3 class counts are shown next to the counts produced by this split.

In [5]:
train_counts = data.class_count_table(splits["train"], "train")
validation_counts = data.class_count_table(splits["validation"], "validation")
display(pd.concat([train_counts, validation_counts], ignore_index=True).round(2))

print(f"Train rows: {len(splits['train']):,} (paper: 11,293)")
print(f"Validation rows: {len(splits['validation']):,} (paper: 1,255)")
display(data.compare_to_paper(splits["train"], splits["validation"]))

,split,label,label_id,count,percentage
0,train,NEG,0,4159,36.83
1,train,NEU,1,5172,45.80
2,train,POS,2,1962,17.37
3,validation,NEG,0,462,36.81
4,validation,NEU,1,575,45.82
5,validation,POS,2,218,17.37


Train rows: 11,293 (paper: 11,293)
Validation rows: 1,255 (paper: 1,255)


,split,label,ours,paper,difference
0,train,NEG,4159,4139,20
1,train,NEU,5172,5197,-25
2,train,POS,1962,1957,5
3,train,TOTAL,11293,11293,0
4,validation,NEG,462,482,-20
5,validation,NEU,575,550,25
6,validation,POS,218,223,-5
7,validation,TOTAL,1255,1255,0


## 3. Save the preprocessed splits

Each file has exactly the columns `text,label`. The test file is saved with its labels for the later final evaluation, but those labels are not inspected or used for any decision in this notebook.

In [6]:
paths = data.save_splits(splits, ROOT / config["output_dir"])
for name, path in paths.items():
    print(f"{name}: {path} ({len(splits[name]):,} rows)")

saved_train = pd.read_csv(paths["train"])
print("Saved train columns:", list(saved_train.columns))
print("Saved validation columns:", list(pd.read_csv(paths["validation"]).columns))
print("Saved test columns:", list(pd.read_csv(paths["test"], nrows=0).columns))
print("The test file was preprocessed with the same function and saved; its labels were not used for any decision.")

train: /home/nour/FCAI-H/fourth year last_dance/Sem1/Neural Networks & deep learning/project/data/splits/arsarcasm_v2/train.csv (11,293 rows)
validation: /home/nour/FCAI-H/fourth year last_dance/Sem1/Neural Networks & deep learning/project/data/splits/arsarcasm_v2/validation.csv (1,255 rows)
test: /home/nour/FCAI-H/fourth year last_dance/Sem1/Neural Networks & deep learning/project/data/splits/arsarcasm_v2/test.csv (3,000 rows)


Saved train columns: ['text', 'label']
Saved validation columns: ['text', 'label']
Saved test columns: ['text', 'label']
The test file was preprocessed with the same function and saved; its labels were not used for any decision.


## 4. Token statistics before and after preprocessing

The same rows are compared before and after preprocessing: the 11,293 training rows and the 3,000 test rows. `tokens_per_word` is the aggregate ratio (total AraBERT tokens divided by total whitespace tokens). `mean_tweet_tokens_per_word` is the mean of the per-tweet ratios.

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(eda.ARABERT_MODEL_NAME)

stage_specs = [
    ("before", "train", splits["raw_train"], "tweet"),
    ("before", "test", splits["raw_test"], "tweet"),
    ("after", "train", splits["train"], "text"),
    ("after", "test", splits["test"], "text"),
]

rows = []
for stage, split_name, frame, text_column in stage_specs:
    stats = eda.token_statistics(frame, text_column, tokenizer, max_length=256)
    rows.append(
        {
            "stage": stage,
            "split": split_name,
            "rows": stats["n_texts"],
            "over_256": stats["over_limit_count"],
            "over_256_pct": stats["over_limit_percentage"],
            "mean_tokens": stats["mean_tokens"],
            "mean_words": stats["mean_words"],
            "mean_tweet_tokens_per_word": stats["mean_tweet_tokens_per_word"],
            "tokens_per_word": stats["tokens_per_word"],
        }
    )

token_stats = pd.DataFrame(rows)
display(token_stats.round(3))

/home/nour/py/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,stage,split,rows,over_256,over_256_pct,mean_tokens,mean_words,mean_tweet_tokens_per_word,tokens_per_word
0,before,train,11293,0,0.0,38.828,15.030,3.073,2.583
1,before,test,3000,0,0.0,31.978,16.852,2.611,1.898
2,after,train,11293,0,0.0,26.303,15.717,1.805,1.673
3,after,test,3000,0,0.0,28.125,17.109,1.840,1.644


### Did the train/test tokens-per-word gap shrink?

In [8]:
for column in ("tokens_per_word", "mean_tweet_tokens_per_word"):
    before = token_stats.query("stage == 'before'").set_index("split")[column]
    after = token_stats.query("stage == 'after'").set_index("split")[column]
    before_gap = float(before["train"] - before["test"])
    after_gap = float(after["train"] - after["test"])
    print(f"{column}:")
    print(f"  before gap = {before_gap:.3f}")
    print(f"  after gap  = {after_gap:.3f}")
    print(f"  shrank     = {after_gap < before_gap} (change {after_gap - before_gap:+.3f})")

tokens_per_word:
  before gap = 0.686
  after gap  = 0.030
  shrank     = True (change -0.656)
mean_tweet_tokens_per_word:
  before gap = 0.462
  after gap  = -0.035
  shrank     = True (change -0.497)


## 5. Marker breakdown before and after preprocessing

The same URL, mention, and hashtag patterns from Phase A are recomputed. After preprocessing, URL and mention shares become zero because those spans are replaced by placeholders. Hashtags are not replaced, although Latin/digit hashtags may no longer match after whitespace insertion.

In [9]:
marker_tables = []
for stage, split_name, frame, text_column in stage_specs:
    lengths = eda.tokenizer_lengths(frame[text_column].tolist(), tokenizer)
    table = eda.marker_breakdown(
        frame.reset_index(drop=True),
        lengths,
        text_column=text_column,
    )
    table.insert(0, "stage", stage)
    table.insert(1, "split", split_name)
    marker_tables.append(table)

marker_table = pd.concat(marker_tables, ignore_index=True)
display(marker_table.round(2))

,stage,split,marker,tweet_count,share_pct,tokens_per_word_with,tokens_per_word_without
0,before,train,url,4323,38.28,4.09,1.84
1,before,train,mention,2587,22.91,2.91,2.48
2,before,train,hashtag,5192,45.98,3.02,2.24
3,before,test,url,440,14.67,3.71,1.69
4,before,test,mention,19,0.63,3.44,1.89
5,before,test,hashtag,960,32.00,1.97,1.85
6,after,train,url,0,0.00,NaN,1.67
7,after,train,mention,0,0.00,NaN,1.67
8,after,train,hashtag,4963,43.95,1.89,1.52
9,after,test,url,0,0.00,NaN,1.64


## 6. Final `[UNK]` audit

This pass adds Fitzpatrick skin-tone removal, Arabic Presentation Forms NFKC normalization, and slash replacement. The table below compares the recorded `[UNK]` counts immediately before this pass with the current post-change numbers, for train and validation only (test is never used for this report).

In [10]:
# Recorded [UNK] counts immediately before this final preprocessing pass.
before_pass = pd.DataFrame(
    [
        {"split": "train", "stage": "before this pass", "total_UNK": 1150, "pct_tweets_with_UNK": 4.622, "mean_tokens": 26.285},
        {"split": "validation", "stage": "before this pass", "total_UNK": 174, "pct_tweets_with_UNK": 4.861, "mean_tokens": 26.355},
    ]
)

def unk_summary(texts):
    encoded = tokenizer(
        texts,
        add_special_tokens=True,
        truncation=False,
        padding=False,
        return_attention_mask=False,
    )
    ids = encoded["input_ids"]
    unk_id = tokenizer.unk_token_id
    lengths = [len(row) for row in ids]
    total_unk = sum(row.count(unk_id) for row in ids)
    tweets_with_unk = sum(1 for row in ids if unk_id in row)
    return {
        "total_UNK": total_unk,
        "pct_tweets_with_UNK": 100.0 * tweets_with_unk / len(ids),
        "mean_tokens": sum(lengths) / len(lengths),
    }

after_rows = []
for split_name in ("train", "validation"):
    result = unk_summary(splits[split_name]["text"].astype(str).tolist())
    after_rows.append({"split": split_name, "stage": "after this pass", **result})

before_after = pd.concat([before_pass, pd.DataFrame(after_rows)], ignore_index=True)
display(before_after.round(3))

,split,stage,total_UNK,pct_tweets_with_UNK,mean_tokens
0,train,before this pass,1150,4.622,26.285
1,validation,before this pass,174,4.861,26.355
2,train,after this pass,229,1.311,26.303
3,validation,after this pass,29,1.355,26.386


### Characters that still produce `[UNK]`

The table below lists the 15 most frequent characters in the final preprocessed train and validation text that still tokenize as `[UNK]`, with their Unicode codepoint and name. A character is counted as `[UNK]`-producing if tokenizing it alone returns a single `[UNK]` token. This report is diagnostic only; preprocessing is frozen and is not changed based on it.

In [11]:
from collections import Counter
import unicodedata

char_counts = Counter()
for split_name in ("train", "validation"):
    for text in splits[split_name]["text"].astype(str):
        char_counts.update(text)

oov_rows = []
for character, count in char_counts.items():
    if character.isspace():
        continue
    if tokenizer.tokenize(character) == [tokenizer.unk_token]:
        oov_rows.append(
            {
                "character": character,
                "codepoint": f"U+{ord(character):04X}",
                "name": unicodedata.name(character, "UNKNOWN"),
                "count": count,
            }
        )

oov_table = (
    pd.DataFrame(oov_rows)
    .sort_values("count", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
print(f"Distinct characters that tokenize alone as [UNK]: {len(oov_rows)}")
display(oov_table)

Distinct characters that tokenize alone as [UNK]: 76


,character,codepoint,name,count
0,گ,U+06AF,ARABIC LETTER GAF,42
1,•,U+2022,BULLET,35
2,ڤ,U+06A4,ARABIC LETTER VEH,15
3,چ,U+0686,ARABIC LETTER TCHEH,14
4,ی,U+06CC,ARABIC LETTER FARSI YEH,14
5,ٰ,U+0670,ARABIC LETTER SUPERSCRIPT ALEF,12
6,‼,U+203C,DOUBLE EXCLAMATION MARK,11
7,×,U+00D7,MULTIPLICATION SIGN,7
8,&,U+0026,AMPERSAND,7
9,﴿,U+FD3F,ORNATE RIGHT PARENTHESIS,6


## 7. Summary

- **Preprocessing:** 29/29 sanity checks passed. The single `preprocess_text` function is used for train, validation, and test. Preprocessing is now frozen.
- **Split sizes:** train **11,293**, validation **1,255**, test **3,000**. The train/validation totals match the paper exactly.
- **Label mapping:** `NEG=0`, `NEU=1`, `POS=2`.
- **Class counts:** train NEG 4,159 / NEU 5,172 / POS 1,962; validation NEG 462 / NEU 575 / POS 218. The paper reports train 4,139 / 5,197 / 1,957 and validation 482 / 550 / 223. The split is stratified on sentiment only. The paper's validation counts deviate by about 20 to 25 rows from the sentiment-stratified expectation, which suggests (inference, not fact) that the paper did not stratify on sentiment. Text-only checks find 13 of the 1,255 validation texts also appear in train (6 with a different label), and 6 test rows (5 distinct texts) appear in train or validation.
- **Duplicates:** the 25 training tweets with conflicting sentiment labels and all other duplicates are kept. The stratified split may place conflicting copies on both train and validation sides.
- **No rows dropped:** no preprocessed tweet is empty.
- **256-token limit:** 0 training and 0 test tweets exceed 256 AraBERT tokens both before and after preprocessing.
- **Tokens-per-word gap:** aggregate train/test ratio goes from **2.58 vs 1.90** (gap 0.686) to **1.67 vs 1.64** (gap 0.030). The mean per-tweet ratio goes from **3.07 vs 2.61** (gap 0.462) to **1.81 vs 1.84** (gap -0.035). The gap shrinks substantially under both definitions.
- **Why it shrinks:** before preprocessing, train has far more URLs (38.28% vs 14.67%) and mentions (22.91% vs 0.63%). Those spans have high tokens-per-word ratios and are replaced by single placeholders. This is consistent with the gap reduction. After preprocessing, URL and mention shares are zero, and hashtag shares are 43.95% train vs 27.80% test. Both the numerator (AraBERT tokens) and the denominator (whitespace words) change because inserted spaces raise whitespace word counts.
- **Tashkeel/tatweel:** stripping is applied inside `preprocess_text`. The recorded before/after comparison is total `[UNK]` **4,755 → 1,136** on train and **579 → 172** on validation; tweets with at least one `[UNK]` **19.1% → 4.5%** on train and **18.1% → 4.7%** on validation.
- **Final preprocessing pass:** adding Fitzpatrick removal, presentation-form NFKC, and slash replacement reduces total `[UNK]` from **1,150 → 229** on train and from **174 → 29** on validation. Tweets with at least one `[UNK]` fall from **4.622% → 1.311%** on train and from **4.861% → 1.355%** on validation; mean tokens per tweet changes from **26.285 → 26.303** and from **26.355 → 26.386**.
- **Remaining `[UNK]` characters:** after the final changes, 76 distinct characters still tokenize alone as `[UNK]`; the most frequent are `گ` (U+06AF), `•` (U+2022), `ڤ` (U+06A4), `ی` (U+06CC), and `چ` (U+0686). The top-15 table is in section 6.
- **Rows changed:** versus the pre-pass splits, text changed in **385 / 11,293** train, **45 / 1,255** validation, and **79 / 3,000** test rows; labels are unchanged.
- **Test isolation:** the test file was only preprocessed and saved; its labels were never used for splitting, tuning, or model selection.